# Bài 11 · Mạng tích chập (CNN): một bộ lọc 3×3 trượt khắp chữ số viết tay

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/11-cnn.ipynb)

Notebook đi kèm [Bài 11 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/cnn/) (demo tương tác nằm ở trang bài học).
MLP của các bài trước nối mỗi nơ-ron với cả 784 điểm ảnh và không biết hai điểm ảnh nào nằm cạnh nhau.
Ta tự viết bằng numpy phép tích chập, chạy chính mạng CNN của demo (8 bộ lọc 3×3 → ReLU → max-pooling → lớp dense)
trên 10.000 ảnh MNIST thật, và kiểm tra từng ô một xem “vật dời thì bản đồ dời theo” đúng đến đâu.

**Bạn sẽ làm:**
1. Viết phép tích chập bằng vòng lặp trên một chữ số, rồi vector hóa nó thành một phép nhân ma trận $Z = PW + b$.
2. Dùng bộ lọc cạnh dọc, cạnh ngang, rồi đọc 8 bộ lọc mà CNN của demo tự học.
3. Chạy CNN của demo trên tập kiểm tra, đếm tham số và phép nhân–cộng so với MLP, rồi tự huấn luyện một CNN.
4. Kiểm tra tính đẳng biến từng ô, rồi dời cả 10.000 ảnh để thấy CNN chưa bất biến.
5. Xáo trộn vị trí điểm ảnh, xem MLP không hề hay biết còn CNN mất lợi thế; rồi nhìn vùng tiếp nhận để sang Bài 12.

**Cần biết trước:** [Bài 03 · Mạng nhiều lớp & bài toán XOR](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mlp-va-xor/)
(lớp ẩn, $z = xW + b$) và [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/).
**Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) và tệp kết quả của demo lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, hoặc đọc từ đúng tệp kết quả mà demo dùng,
và trùng với số trong demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from numpy.lib.stride_tricks import sliding_window_view

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
SEQUENTIAL = LinearSegmentedColormap.from_list("trang_xanh", ["#ffffff", BLUE])  # 0 → lớn
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38."""
    s = (
        f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    )  # không in “−0”
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def sci(x):
    """Số rất nhỏ, dạng khoa học: 2,4e−08."""
    return f"{x:.1e}".replace(".", ",").replace("-", "−")


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def vn_axes(fig):
    """Đổi nhãn vạch của mọi trục tuyến tính (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(vn_tick))


print(vn(13610, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.9798), sci(2.4e-8))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998) gồm 70.000 ảnh chữ số viết tay xám 28×28: 60.000 ảnh huấn luyện, 10.000 ảnh kiểm tra.
Ô dưới tải 4 tệp gốc từ một bản sao công khai và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu.

Ô dưới cũng tải `mnist-cnn.json`, tệp mà demo của bài đọc. Nó do chương trình `reel/mnist_mlp/cnn.py` của khóa học
tạo ra: huấn luyện CNN 3 lần (seed 0, 1, 2) và vài mạng so sánh, rồi thử từng mạng trên ảnh kiểm tra bị dời.
Toàn bộ việc đó lâu hơn nhiều so với một notebook. Tệp chứa **trọng số của CNN seed 0** (đúng mạng
mà demo chạy trong trình duyệt) và các độ chính xác đã đo. Ta sẽ tự chạy lại mạng đó, và chỉ đọc sẵn những kết quả
cần huấn luyện lâu.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web).
# build.py thay COURSE_DATA_URL bằng thư mục data/ của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
demo = fetch_json("mnist-cnn.json", "d4c7c4c978f9abf403a2193fd907dcd7")
n_tr, n_te = len(train_lab), len(test_lab)
print(
    f"{vn(n_tr, 0)} ảnh huấn luyện + {vn(n_te, 0)} ảnh kiểm tra = {vn(n_tr + n_te, 0)};  mỗi ảnh", train_img.shape[1:]
)
print("trong mnist-cnn.json:", ", ".join(demo))

Ta chia mọi điểm ảnh cho 255 để có số trong $[0, 1]$: 0 là nền, 1 là nét đậm nhất. Khác các bài trước, ảnh được giữ
nguyên dạng **lưới 28×28** thay vì trải thành vectơ 784 số, vì tích chập cần biết điểm ảnh nào nằm cạnh điểm ảnh nào.

In [ ]:
Xtr, ytr = train_img / 255.0, train_lab.astype(int)  # 60.000 × 28 × 28
Xte, yte = test_img / 255.0, test_lab.astype(int)  # 10.000 × 28 × 28
SAMPLES = [s["mnist"] for s in demo["samples"]]  # 10 ảnh kiểm tra mà demo cho chọn, mỗi chữ số một ảnh
print("ảnh mẫu của demo (chỉ số trong tập kiểm tra):", SAMPLES)
print("nhãn của chúng:", [int(yte[i]) for i in SAMPLES])
ink_s = test_img[SAMPLES] > 0
margin = min(
    min(np.argmax(ink_s.any(axis=a), axis=1).min(), np.argmax(ink_s.any(axis=a)[:, ::-1], axis=1).min()) for a in (1, 2)
)
print(f"lề trống nhỏ nhất của 10 ảnh mẫu: {margin} điểm ảnh")
assert margin >= 4

Hình dưới là 10 ảnh đó, đúng thứ tự trong demo (mỗi chữ số một ảnh, đều có lề trống ít nhất 4 điểm ảnh mỗi bên,
nên dời tới 4 điểm ảnh không cắt mất nét). Tiêu đề là chỉ số trong tập kiểm tra và nhãn đúng.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.5))
for ax, i in zip(axes, SAMPLES):
    ax.imshow(Xte[i], cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(f"#{i} · số {yte[i]}", fontsize=9)
    ax.axis("off")
plt.show()

## 3 · Ảnh là một ma trận số; bộ lọc 3×3 trượt trên nó

Với máy tính, chữ số 4 (#109) chỉ là một ma trận 28×28. Ô dưới in một góc của nó bằng byte gốc 0–255
(0 là nền, 255 là mực đậm nhất).

In [ ]:
I = Xte[109]  # chữ số 4 trong 10 ảnh mẫu của demo
print("ma trận", I.shape, "; hàng 8–14, cột 4–13 (byte gốc):")
print(test_img[109][8:15, 4:14])

Một **bộ lọc** (kernel) $K$ là một ma trận nhỏ $k \times k$, ở đây $3 \times 3$. Đặt nó lên một cửa sổ $3 \times 3$
của ảnh, nhân từng cặp phần tử chồng lên nhau rồi cộng 9 tích lại (cùng độ lệch $b$): đó là một ô của
**bản đồ đặc trưng** (feature map). Trượt cửa sổ từng điểm ảnh một trên khắp ảnh, ta được cả bản đồ:

$$z_{ij} = \sum_{m=0}^{k-1}\sum_{n=0}^{k-1} I_{i+m,\,j+n}\, K_{mn} + b .$$

Cửa sổ phải nằm gọn trong ảnh (không đệm thêm số 0 ở mép, bước trượt 1), nên $i, j$ chạy từ 0 đến $28 - 3 = 25$:
bản đồ có $26 \times 26 = 676$ ô. Ta thử bộ lọc **Sobel cạnh dọc**, cột phải trừ cột trái:

$$K_\text{dọc} = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}.$$

Ô dưới tính một ô bằng tay, ở cửa sổ có hàng trên cùng là hàng 12 và cột trái cùng là cột 6.

In [ ]:
K_V = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)  # Sobel cạnh dọc: cột phải trừ cột trái
K_H = K_V.T.copy()  # Sobel cạnh ngang: hàng dưới trừ hàng trên
i0, j0 = 12, 6
window = I[i0 : i0 + 3, j0 : j0 + 3]
print("cửa sổ 3×3 của ảnh:\n", np.round(window, 2))
print("9 tích I·K:\n", np.round(window * K_V, 2))
print(f"tổng 9 tích = z({i0}, {j0}) = {vn((window * K_V).sum(), 4)}")

Cột trái của cửa sổ gần như là nền, cột phải là mực, nên cột phải trừ cột trái cho một số dương lớn: cửa sổ này nằm
trên **mép trái** của một nét dọc. Giờ viết cả phép trượt bằng vòng lặp, đếm từng phép nhân–cộng (MAC):

In [ ]:
def conv2d_loop(I, K, b=0.0):
    """Tích chập 'valid', bước trượt 1, viết thẳng theo công thức. Trả về bản đồ và số phép nhân–cộng."""
    k = K.shape[0]
    rows, cols = I.shape[0] - k + 1, I.shape[1] - k + 1
    Z, macs = np.zeros((rows, cols)), 0
    for i in range(rows):
        for j in range(cols):
            s = b
            for m in range(k):
                for n in range(k):
                    s += I[i + m, j + n] * K[m, n]  # một phép nhân–cộng
                    macs += 1
            Z[i, j] = s
    return Z, macs


t0 = time.perf_counter()
Z_loop, macs = conv2d_loop(I, K_V)
t_loop = time.perf_counter() - t0
print(f"bản đồ {Z_loop.shape}, {vn(macs, 0)} MAC; z({i0}, {j0}) = {vn(Z_loop[i0, j0], 4)}")
peak = tuple(int(v) for v in np.unravel_index(Z_loop.argmax(), Z_loop.shape))
print(f"lớn nhất {vn(Z_loop.max(), 4)} ở ô {peak}, nhỏ nhất {vn(Z_loop.min(), 4)}")
assert macs == 676 * 9 == 6084

### Vector hóa: mỗi cửa sổ là một vectơ hàng

Trải mỗi cửa sổ $3 \times 3$ thành một vectơ hàng 9 số. Xếp 676 cửa sổ chồng lên nhau, ta được ma trận
$P$ cỡ $676 \times 9$ (cách làm này gọi là **im2col**). Bộ lọc trải thành cột 9 số, và cả bản đồ là **một phép nhân
ma trận**, đúng quy ước $z = xW + b$ của khóa học, chỉ khác là mỗi “ảnh” $x$ giờ là một cửa sổ 9 điểm ảnh:

$$Z = P\,W + b, \qquad P \in \mathbb{R}^{676 \times 9},\; W \in \mathbb{R}^{9 \times F}.$$

Với $F$ bộ lọc, $W$ có $F$ cột và ta được $F$ bản đồ cùng lúc. Hàm `sliding_window_view` của numpy tạo $P$ mà không
chép dữ liệu (mỗi điểm ảnh xuất hiện trong tới 9 cửa sổ). Hàm dưới nhận một ảnh hoặc cả lô ảnh.

In [ ]:
def patches(X, k=3):
    """Mọi cửa sổ k×k của ảnh (hoặc lô ảnh) X, trải thành vectơ hàng: (..., H−k+1, W−k+1, k·k)."""
    *lead, H, W = X.shape
    return sliding_window_view(X, (k, k), axis=(-2, -1)).reshape(*lead, H - k + 1, W - k + 1, k * k)


def conv2d(X, W, b=0.0):
    """Tích chập với F bộ lọc: W cỡ k·k × F (hàng m·k + n ↔ phần tử K[m, n]); trả về (..., H−k+1, W−k+1, F)."""
    return patches(X, int(np.sqrt(W.shape[0]))) @ W + b


P1 = patches(I)
t0 = time.perf_counter()
Z_vec = conv2d(I, K_V.reshape(9, 1))[..., 0]
t_vec = time.perf_counter() - t0
print("P của một ảnh:", P1.reshape(-1, 9).shape, " ·  hàng của cửa sổ (12, 6):", np.round(P1[i0, j0], 2))
print(
    f"vòng lặp {vn(t_loop * 1000, 1)} ms, nhân ma trận {vn(t_vec * 1000, 3)} ms; lệch lớn nhất {sci(np.abs(Z_vec - Z_loop).max())}"
)
assert np.allclose(Z_vec, Z_loop, atol=1e-12)

### “Tích chập” trong deep learning thật ra là tương quan chéo

Trong toán học và xử lý tín hiệu, tích chập **lật** bộ lọc trước khi trượt: $\sum_{m,n} I_{i+m,j+n} K_{k-1-m,\,k-1-n}$.
Công thức ở trên không lật, nên đúng tên của nó là **tương quan chéo** (cross-correlation). Các thư viện deep learning
đều tính tương quan chéo nhưng vẫn gọi là tích chập. Với bộ lọc học được thì chuyện lật không quan trọng: một mạng
dùng tích chập thật chỉ học ra đúng bộ lọc đó, lật ngược. Numpy có cả hai phép cho tín hiệu 1 chiều, nên ta kiểm tra được:

In [ ]:
row = I[14]  # hàng 14 của chữ số 4
k1 = np.array([1.0, 2.0, -3.0])  # một bộ lọc 1 chiều không đối xứng
same = np.allclose(np.convolve(row, k1, "valid"), np.correlate(row, k1[::-1], "valid"))
print("np.convolve(x, k) = np.correlate(x, k lật ngược):", same)
print("Sobel dọc lật cả hai chiều = −Sobel dọc:", np.array_equal(K_V[::-1, ::-1], -K_V))
assert same

Vậy nếu dùng tích chập “thật”, bản đồ của bộ lọc Sobel dọc chỉ đổi dấu. Từ đây ta gọi theo thói quen chung là tích chập.

Hình dưới: bên trái là chữ số 4 với cửa sổ $3 \times 3$ vừa tính (khung xanh lá); bên phải là cả bản đồ $26 \times 26$
của bộ lọc cạnh dọc (xanh dương là dương, cam là âm, trắng là 0), ô vừa tính đánh dấu bằng khung xanh lá.
Ô $(i, j)$ của bản đồ ứng với cửa sổ có góc trên bên trái ở $(i, j)$, tức tâm ở điểm ảnh $(i+1, j+1)$.

In [ ]:
def show_map(ax, Z, title, vmax=None):
    vmax = vmax or np.abs(Z).max() or 1
    im = ax.imshow(Z, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return im


fig, (a, b) = plt.subplots(1, 2, figsize=(7.4, 3.6))
a.imshow(I, cmap="gray_r", vmin=0, vmax=1)
a.add_patch(Rectangle((j0 - 0.5, i0 - 0.5), 3, 3, fill=False, ec=GREEN, lw=2.5))
a.set_title("ảnh 28×28, cửa sổ ở (12, 6)", fontsize=10)
a.axis("off")
im = show_map(b, Z_vec, "bản đồ cạnh dọc 26×26")
b.add_patch(Rectangle((j0 - 0.5, i0 - 0.5), 1, 1, fill=False, ec=GREEN, lw=2.5))
fig.colorbar(im, ax=b, fraction=0.046)
vn_axes(fig)
plt.show()

## 4 · Bộ lọc phát hiện cạnh

Bộ lọc cạnh dọc cho số dương ở **mép trái** của mỗi nét dọc (cửa sổ đi từ nền 0 sang mực 1) và số âm ở mép phải
(từ mực về nền); ở giữa một vùng phẳng, cột phải và cột trái bằng nhau nên ra 0. Đổi vai trò hàng và cột, ta được
**Sobel cạnh ngang** (hàng dưới trừ hàng trên): dương ở mép trên của một nét ngang, âm ở mép dưới.

Hai bộ lọc là hai cột của $W$ (cỡ $9 \times 2$), nên một phép nhân ma trận cho ra cả hai bản đồ. Hình dưới vẽ
chữ số 4 và hai bản đồ cùng một thang màu (xanh dương: dương, cam: âm).

In [ ]:
W_sobel = np.stack([K_V.ravel(), K_H.ravel()], axis=1)  # 9 × 2: mỗi cột một bộ lọc
Z2 = conv2d(I, W_sobel)  # 26 × 26 × 2
vmax = np.abs(Z2).max()
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.4))
axes[0].imshow(I, cmap="gray_r", vmin=0, vmax=1)
axes[0].set_title("chữ số 4 (#109)", fontsize=10)
axes[0].axis("off")
show_map(axes[1], Z2[..., 0], "Sobel cạnh dọc", vmax)
im = show_map(axes[2], Z2[..., 1], "Sobel cạnh ngang", vmax)
fig.colorbar(im, ax=axes[2], fraction=0.046)
vn_axes(fig)
plt.show()
for name, Z in zip(["dọc", "ngang"], np.moveaxis(Z2, -1, 0)):
    print(f"cạnh {name:>5}: lớn nhất {vn(Z.max(), 2)}, nhỏ nhất {vn(Z.min(), 2)}, {(np.abs(Z) > 1).sum()} ô có |z| > 1")

Chỉ những cửa sổ nằm trên mép nét mới cho $|z|$ lớn; nền trắng và lòng nét đều cho 0. Bản đồ cạnh dọc sáng dọc
theo hai nét đứng của chữ 4 (nét chéo bên trái, nét dọc bên phải), bản đồ cạnh ngang sáng ở thanh ngang. Một bộ lọc $3 \times 3$ là một **mẫu cục bộ**, và bản đồ
cho biết mẫu đó xuất hiện ở đâu trong ảnh.

## 5 · Nhiều bộ lọc → nhiều bản đồ: CNN của demo

CNN trong demo không dùng bộ lọc thiết kế sẵn. Nó có $F = 8$ bộ lọc $3 \times 3$ với trọng số ban đầu ngẫu nhiên,
và **học** chúng cùng lúc với phần còn lại của mạng bằng lan truyền ngược (Bài 06), theo đúng công thức của MLP:
SGD, $\eta = 0{,}1$, lô 64 ảnh, 5 lượt (epoch) trên 60.000 ảnh. Ô dưới đọc trọng số seed 0 từ tệp của demo
(lớp dense lưu ở dạng float32 mã hóa base64) và in kích thước từng phần.

In [ ]:
CNN = {
    "Wc": np.array(demo["conv"]["W"]),  # 9 × 8: mỗi cột là một bộ lọc 3×3 đã trải phẳng
    "bc": np.array(demo["conv"]["b"]),  # 8 độ lệch, mỗi bộ lọc một số
    "Wd": np.frombuffer(base64.b64decode(demo["dense"]["W"]), "<f4").astype(float).reshape(-1, 10),  # 1.352 × 10
    "bd": np.array(demo["dense"]["b"]),
}
for name, v in CNN.items():
    print(f"{name}: {v.shape}")
F = CNN["Wc"].shape[1]
assert (demo["k"], F) == (3, 8)

Hình dưới: hàng trên là 8 bộ lọc học được (mỗi ô vuông 3×3 là một trọng số; xanh dương dương, cam âm), hàng dưới là
8 bản đồ đặc trưng của chữ số 4 sau ReLU (trắng là 0, xanh càng đậm càng lớn). Dưới mỗi bộ lọc là độ giống (cosin) của
nó với Sobel dọc (“d”) và Sobel ngang (“n”): ±1 là giống hệt (hoặc ngược dấu), 0 là không liên quan.

In [ ]:
def cos(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))


maps = np.maximum(conv2d(I, CNN["Wc"], CNN["bc"]), 0)  # 26 × 26 × 8, sau ReLU
fig, axes = plt.subplots(2, F, figsize=(10, 3.2))
for f in range(F):
    w = CNN["Wc"][:, f]
    show_map(axes[0, f], w.reshape(3, 3), f"#{f + 1}\nd {vn(cos(w, K_V.ravel()), 2)} · n {vn(cos(w, K_H.ravel()), 2)}")
    axes[1, f].imshow(maps[..., f], cmap=SEQUENTIAL, vmin=0)
    axes[1, f].axis("off")
plt.tight_layout()
plt.show()
for f in range(F):
    w = CNN["Wc"][:, f]
    print(
        f"bộ lọc #{f + 1}: b = {vn(CNN['bc'][f], 3):>6}, cos với Sobel dọc {vn(cos(w, K_V.ravel()), 2):>5}, "
        f"ngang {vn(cos(w, K_H.ravel()), 2):>5}"
    )

Không ai thiết kế 8 bộ lọc này, nhưng vài bộ lọc rất giống bộ dò cạnh: #4 gần như là Sobel dọc (cosin 0,93),
#6 là Sobel ngang ngược dấu (−0,87), #1 nhạy với cạnh chéo. Bộ lọc #5 có độ lệch dương nên sáng cả ở nền,
những bộ lọc có $b$ âm chỉ sáng ở chỗ có nét.

Chúng bắt đầu từ đâu? `cnn.py` khởi tạo bộ lọc bằng phân phối He, $\mathcal{N}(0, 2/9)$, từ seed 0; đó là lần rút
ngẫu nhiên đầu tiên, nên ta dựng lại được đúng 8 bộ lọc ban đầu và so với 8 bộ lọc sau 5 lượt học.

In [ ]:
Wc0 = np.random.default_rng(0).normal(0, np.sqrt(2 / 9), (9, F))  # lần rút đầu tiên của cnn.py với seed 0


def edge_like(w):
    """|cosin| lớn nhất với Sobel dọc hoặc Sobel ngang: 1 là một bộ dò cạnh hoàn hảo (kể cả ngược dấu)."""
    return max(abs(cos(w, K_V.ravel())), abs(cos(w, K_H.ravel())))


for f in range(F):
    w0, w1 = Wc0[:, f], CNN["Wc"][:, f]
    print(
        f"#{f + 1}: cos(ban đầu, đã học) = {vn(cos(w0, w1), 2)};  giống cạnh {vn(edge_like(w0), 2)} → {vn(edge_like(w1), 2)};"
        f"  độ dài {vn(np.linalg.norm(w0), 2)} → {vn(np.linalg.norm(w1), 2)}"
    )
n0, n1 = (sum(edge_like(W[:, f]) >= 0.75 for f in range(F)) for W in (Wc0, CNN["Wc"]))
grow = np.linalg.norm(CNN["Wc"], axis=0) / np.linalg.norm(Wc0, axis=0)
print(f"số bộ lọc giống bộ dò cạnh (|cos| ≥ 0,75): ban đầu {n0}/8, sau khi học {n1}/8")
print(f"độ dài tăng {vn(grow.min(), 2)} đến {vn(grow.max(), 2)} lần (trung vị {vn(np.median(grow), 2)})")

Một điều bất ngờ: mỗi bộ lọc học được vẫn **cùng hướng với chính nó lúc khởi tạo** (cosin 0,70 đến 0,95). Việc học
chủ yếu kéo dài chúng ra (độ dài tăng 1,05 đến 2,62 lần) và xoay nhẹ về phía bộ dò cạnh: số bộ lọc có $|\cos| \ge 0{,}75$
với Sobel tăng từ 1 lên 4. Với một lớp tích chập và 9 trọng số, một bộ lọc ngẫu nhiên đã là một bộ dò mẫu cục bộ
dùng được; mục 12 sẽ đo xem học bộ lọc giúp được bao nhiêu.

## 6 · Cả mạng: ReLU → max-pooling → lớp dense

Sau tích chập, mạng của demo làm ba việc:
1. **ReLU** từng ô của 8 bản đồ $26 \times 26$.
2. **Max-pooling 2×2**: chia mỗi bản đồ thành các khối $2 \times 2$ không chồng nhau và giữ số lớn nhất của mỗi khối,
   nên $26 \times 26 \times 8$ còn $13 \times 13 \times 8$. Pooling là nội dung chính của Bài 12; ở đây chỉ cần biết
   nó làm bản đồ nhỏ đi 4 lần.
3. Trải $13 \cdot 13 \cdot 8 = 1.352$ số thành một vectơ hàng $h$ và tính **10 logit** $z = hW_d + b_d$ bằng một lớp
   dense như mọi bài trước.

Ô dưới viết cả mạng (chạy theo lô 200 ảnh: mảng trung gian nhỏ thì nhanh hơn), in kích thước dữ liệu sau từng bước cho một ảnh,
rồi chạy trên cả 10.000 ảnh kiểm tra.

In [ ]:
def maxpool2(A):
    """Max-pooling 2×2, bước 2, trên hai trục không gian của (n, H, W, F)."""
    return np.maximum(np.maximum(A[:, 0::2, 0::2], A[:, 0::2, 1::2]), np.maximum(A[:, 1::2, 0::2], A[:, 1::2, 1::2]))


def cnn_logits(P, X, chunk=200, verbose=False):
    out = []
    for i in range(0, len(X), chunk):
        A = np.maximum(conv2d(X[i : i + chunk], P["Wc"], P["bc"]), 0)  # tích chập + ReLU: n × 26 × 26 × 8
        H = maxpool2(A)  # n × 13 × 13 × 8
        h = H.reshape(len(H), -1)  # n × 1.352, thứ tự (hàng, cột, bộ lọc) như cnn.py
        out.append(h @ P["Wd"] + P["bd"])
        if verbose:
            print(
                "ảnh",
                X[i : i + chunk].shape,
                "→ bản đồ",
                A.shape,
                "→ pooling",
                H.shape,
                "→",
                h.shape,
                "→ logit",
                out[-1].shape,
            )
    return np.concatenate(out)


z109 = cnn_logits(CNN, Xte[109:110], verbose=True)[0]
print(f"ảnh #109 (số {yte[109]}): CNN đoán {z109.argmax()}")
Z_cnn = cnn_logits(CNN, Xte)
acc_cnn = (Z_cnn.argmax(1) == yte).mean()
SHIFTS = [(dx, dy) for dy in range(-4, 5) for dx in range(-4, 5)]  # thứ tự của lưới dời 9 × 9 trong tệp
CENTER = SHIFTS.index((0, 0))
print(f"CNN đúng {vn((Z_cnn.argmax(1) == yte).sum(), 0)}/10.000 ảnh kiểm tra = {pct(acc_cnn)}")
print(
    f"tệp của demo ghi: {pct(demo['grid']['cnn'][CENTER])}; MLP của các bài trước: {pct(demo['grid']['mlp'][CENTER])}"
)
assert abs(acc_cnn - demo["grid"]["cnn"][CENTER]) < 5e-4, "khác với demo: hãy kiểm tra lại dữ liệu"
assert all(Z_cnn[i].argmax() == yte[i] for i in SAMPLES)  # demo chọn 10 ảnh mà cả CNN lẫn MLP đều đoán đúng

Mạng chạy trong notebook cho đúng 97,98% như demo (cùng trọng số, cùng phép tính), cao hơn MLP 784 → 128 → 64 → 10
của các bài trước (97,27%), dù học cùng công thức: SGD, $\eta = 0{,}1$, lô 64, 5 lượt.

### 🤔 Dự đoán trước
Lớp tích chập có 72 trọng số (8 bộ lọc × 9), lớp dense có 13.520 ($1.352 \times 10$). Với một ảnh, lớp nào tốn
nhiều **phép nhân–cộng** hơn? Và cả CNN có ít phép tính hơn MLP không, khi nó có ít tham số hơn khoảng 8 lần?

In [ ]:
SIZES = demo["mlpSizes"]  # [784, 128, 64, 10]: MLP của các bài trước
mlp_params = sum((n_in + 1) * n_out for n_in, n_out in itertools.pairwise(SIZES))
mlp_macs = sum(n_in * n_out for n_in, n_out in itertools.pairwise(SIZES))
M, Q = 28 - 3 + 1, (28 - 3 + 1) // 2  # 26: cạnh bản đồ; 13: sau pooling
conv_params, conv_macs = (9 + 1) * F, M * M * 9 * F  # mỗi ô của mỗi bản đồ: 9 phép nhân–cộng
dense_params, dense_macs = (Q * Q * F + 1) * 10, Q * Q * F * 10
print(f"tích chập: {vn(conv_params, 0):>6} tham số, {vn(conv_macs, 0):>7} MAC  (72 trọng số dùng lại ở {M * M} vị trí)")
print(f"dense    : {vn(dense_params, 0):>6} tham số, {vn(dense_macs, 0):>7} MAC")
print(f"CNN      : {vn(conv_params + dense_params, 0):>6} tham số, {vn(conv_macs + dense_macs, 0):>7} MAC")
print(f"MLP      : {vn(mlp_params, 0):>6} tham số, {vn(mlp_macs, 0):>7} MAC")
print(
    f"CNN ít tham số hơn {vn(mlp_params / (conv_params + dense_params), 2)} lần, ít MAC hơn {vn(mlp_macs / (conv_macs + dense_macs), 2)} lần;"
    f" tích chập tốn MAC gấp {vn(conv_macs / dense_macs, 2)} lần lớp dense"
)
assert (conv_params, conv_macs, dense_params, dense_macs) == (80, 48672, 13530, 13520)
assert (mlp_params, mlp_macs) == (109386, 109184)

Lớp tích chập chỉ có 80 tham số (72 trọng số + 8 độ lệch) nhưng tốn **48.672** phép nhân–cộng mỗi ảnh, gấp 3,6 lần
lớp dense có 13.530 tham số. Mỗi trọng số tích chập được dùng lại ở 676 vị trí, nên với tích chập, số tham số và số
phép tính là hai chuyện khác nhau. Cả CNN vẫn rẻ hơn MLP: ít tham số hơn 8 lần, ít phép tính hơn 1,76 lần.

Chia sẻ trọng số tiết kiệm bao nhiêu? Nếu mỗi vị trí có bộ trọng số riêng (lớp “kết nối cục bộ”, *locally connected*),
8 bản đồ sẽ cần $676 \cdot 8 \cdot (9 + 1)$ tham số. Với ảnh lớn, khác biệt còn lớn hơn: lớp đầu của AlexNet
(ảnh 227×227×3, 96 bộ lọc 11×11×3, bước 4, bản đồ 55×55) là ví dụ của khóa CS231n.

In [ ]:
print(f"8 bản đồ không chia sẻ trọng số: {vn(M * M * F * (9 + 1), 0)} tham số, thay vì {conv_params}")
side = (227 - 11) // 4 + 1  # công thức (W − F)/S + 1 của CS231n
alex_local = side * side * 96 * (11 * 11 * 3 + 1)
alex_shared = 96 * (11 * 11 * 3 + 1)
print(
    f"AlexNet lớp 1: bản đồ {side}×{side}×96; không chia sẻ {vn(alex_local, 0)} tham số, chia sẻ {vn(alex_shared, 0)}"
)
print(f"một ảnh màu 224×224 có {vn(224 * 224 * 3, 0)} giá trị; một nơ-ron dense nối với tất cả cần ngần ấy trọng số")

## 7 · Học bộ lọc: gradient của một trọng số dùng chung

Bộ lọc được học bằng lan truyền ngược như mọi trọng số khác (Bài 06). Vì $Z = PW + b$ chỉ là một phép nhân ma trận,
gradient cũng có dạng quen thuộc: với $\partial\mathcal{L}/\partial Z$ đã biết,

$$\frac{\partial \mathcal{L}}{\partial W} = P^\top \frac{\partial \mathcal{L}}{\partial Z}, \qquad
  \frac{\partial \mathcal{L}}{\partial b} = \sum_\text{mọi vị trí} \frac{\partial \mathcal{L}}{\partial Z}.$$

$P^\top$ có 676 cột cho mỗi ảnh, nên gradient của mỗi trọng số bộ lọc là **tổng đóng góp của cả 676 vị trí** (và của
mọi ảnh trong lô): trọng số dùng chung thì gradient cũng cộng chung. Qua max-pooling, gradient chỉ đi về ô lớn nhất
của mỗi khối $2 \times 2$; qua ReLU, chỉ về ô dương. Hàm dưới làm đúng các bước của `cnn.py`: khởi tạo He
($\mathcal{N}(0, 2/9)$ cho bộ lọc, $\mathcal{N}(0, 1/1.352)$ cho lớp dense), SGD, lô 64.

In [ ]:
CORNERS = [(0, 0), (0, 1), (1, 0), (1, 1)]  # 4 ô của một khối 2×2


def init_cnn(rng, F=8):
    return {
        "Wc": rng.normal(0, np.sqrt(2 / 9), (9, F)),
        "bc": np.zeros(F),
        "Wd": rng.normal(0, np.sqrt(1 / (Q * Q * F)), (Q * Q * F, 10)),
        "bd": np.zeros(10),
    }


def softmax(Z):
    E = np.exp(Z - Z.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def cnn_grads(P, x, y, learn_filters=True):
    """Lan truyền xuôi rồi ngược cho một lô; trả về gradient của cross-entropy trung bình theo mọi tham số, và mất mát."""
    pt = patches(x)  # n × 26 × 26 × 9
    A = pt @ P["Wc"] + P["bc"]
    R = np.maximum(A, 0)
    h = maxpool2(R).reshape(len(x), -1)
    dz = softmax(h @ P["Wd"] + P["bd"])
    loss = float(-np.log(dz[np.arange(len(y)), y]).mean())
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    g = {"Wd": h.T @ dz, "bd": dz.sum(0)}
    if learn_filters:
        C = np.stack([R[:, r::2, c::2] for r, c in CORNERS], axis=-1)  # n × 13 × 13 × F × 4
        arg = C.argmax(-1)  # ô nào thắng trong mỗi khối (hòa thì lấy ô đầu, như cnn.py)
        dH = (dz @ P["Wd"].T).reshape(arg.shape)
        dA = np.zeros_like(A)
        for k, (r, c) in enumerate(CORNERS):
            dA[:, r::2, c::2] = dH * (arg == k)  # gradient chỉ đi về ô thắng của khối
        dA *= A > 0  # và chỉ qua những ô ReLU đang bật
        g["Wc"] = pt.reshape(-1, 9).T @ dA.reshape(-1, P["Wc"].shape[1])  # Pᵀ · ∂𝓛/∂Z: tổng trên mọi vị trí
        g["bc"] = dA.reshape(-1, P["Wc"].shape[1]).sum(0)
    return g, loss

Trước khi tin công thức, ta kiểm tra nó bằng **sai phân hữu hạn** như Bài 06: nhích từng trọng số một chút
$\pm\varepsilon$, xem mất mát đổi bao nhiêu, rồi so với gradient vừa tính. Ô dưới làm việc đó cho cả 9 trọng số
của bộ lọc #1 và 5 trọng số của lớp dense, trên một lô 16 ảnh với mạng vừa khởi tạo.

In [ ]:
def loss_of(P, x, y):
    return cnn_grads(P, x, y, learn_filters=False)[1]


P_chk = init_cnn(np.random.default_rng(7))
x_chk, y_chk = Xtr[:16], ytr[:16]
g_chk, _ = cnn_grads(P_chk, x_chk, y_chk)
eps, analytic, numeric = 1e-6, [], []
for name, idx in [("Wc", (t, 0)) for t in range(9)] + [("Wd", (r, 3)) for r in (0, 200, 700, 1000, 1351)]:
    Pp = {k: v.copy() for k, v in P_chk.items()}
    Pm = {k: v.copy() for k, v in P_chk.items()}
    Pp[name][idx] += eps
    Pm[name][idx] -= eps
    numeric.append((loss_of(Pp, x_chk, y_chk) - loss_of(Pm, x_chk, y_chk)) / (2 * eps))
    analytic.append(g_chk[name][idx])
analytic, numeric = np.array(analytic), np.array(numeric)
rel = np.linalg.norm(analytic - numeric) / np.linalg.norm(analytic + numeric)
print("∂𝓛/∂W của bộ lọc #1 (×1.000):", " ".join(vn(1000 * v, 2) for v in g_chk["Wc"][:, 0]))
print(f"sai lệch tương đối so với sai phân hữu hạn (14 trọng số): {sci(rel)}")
assert rel < 1e-6

Gradient khớp sai phân hữu hạn. Giờ huấn luyện thật. Công thức của demo (5 lượt trên 60.000 ảnh) quá lâu cho
notebook này, nên ta chạy **một lượt trên 20.000 ảnh huấn luyện đầu tiên** (313 bước), seed 0, rồi đo trên cả
10.000 ảnh kiểm tra. Hàm `train_cnn` giữ đúng thứ tự rút số ngẫu nhiên của `cnn.py` (khởi tạo, rồi mỗi lượt một hoán vị).

In [ ]:
def train_cnn(seed=0, n=20_000, epochs=1, X=Xtr, y=ytr, eta=0.1, learn_filters=True):
    """SGD lô 64 như reel/mnist_mlp/cnn.py. learn_filters=False: giữ nguyên bộ lọc ngẫu nhiên, chỉ học lớp dense."""
    rng = np.random.default_rng(seed)
    P = init_cnn(rng)
    X, y = X[:n], y[:n]
    losses = []
    for _ in range(epochs):
        order = rng.permutation(len(X))
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            g, loss = cnn_grads(P, X[idx], y[idx], learn_filters)
            for k in g:
                P[k] -= eta * g[k]
            losses.append(loss)
    return P, losses


def accuracy(P, X=Xte, y=yte):
    return float((cnn_logits(P, X).argmax(1) == y).mean())


t0 = time.perf_counter()
P_live, loss_live = train_cnn(seed=0)
acc_live = accuracy(P_live)
print(
    f"{len(loss_live)} bước trên {vn(20_000, 0)} ảnh ({vn(time.perf_counter() - t0, 1)} giây); đúng {pct(acc_live)} trên 10.000 ảnh kiểm tra"
)
full_steps = 5 * len(range(0, 60_000, 64))
print(f"công thức đủ của demo: {vn(full_steps, 0)} bước, gấp {vn(full_steps / len(loss_live), 1)} lần")
print(f"mất mát: 25 bước đầu {vn(np.mean(loss_live[:25]), 3)}, 25 bước cuối {vn(np.mean(loss_live[-25:]), 3)}")
assert len(loss_live) == 313 and abs(acc_live - 0.9266) < 0.003  # số đo lúc viết notebook (aarch64)

Chỉ 313 bước trên một phần ba tập huấn luyện mà CNN đã đúng 92,66%. Công thức đủ của demo dài gấp 15 lần
(4.690 bước). Ô dưới chạy nó nếu bạn đổi `FULL = True` (nó cũng lâu gấp khoảng 15 lần ô trên). Lúc viết notebook, ta đã chạy
và được đúng trọng số trong tệp của demo, trùng từng bit sau khi làm tròn về float32 như `cnn.py`; trên máy khác,
phép cộng làm tròn khác nên có thể lệch ở chữ số cuối.

In [ ]:
FULL = False  # True: 5 lượt × 60.000 ảnh, đúng công thức đã tạo ra mạng của demo
if FULL:
    P_full, _ = train_cnn(seed=0, n=60_000, epochs=5)
    P32 = {k: v.astype(np.float32).astype(float) for k, v in P_full.items()}  # demo lưu trọng số ở float32
    print("lệch lớn nhất so với trọng số của demo:", max(float(np.abs(P32[k] - CNN[k]).max()) for k in CNN))
    print("đúng trên tập kiểm tra:", pct(accuracy(P32)))

## 8 · Đối thủ: MLP của các bài trước

Để so sánh công bằng, ta cần chính MLP 784 → 128 → 64 → 10 mà demo dùng. Huấn luyện lại nó chỉ mất vài giây:
cùng công thức, cùng seed 0 như `reel/mnist_mlp/train.py` (Bài 04 viết lại hàm này từng bước).

In [ ]:
def train_mlp(X, y, seed=0, epochs=5, perm=None):
    """MLP của khóa học, SGD lô 64, η = 0,1. perm: hoán vị các hàng của W₁ lúc khởi tạo (dùng ở mục 11)."""
    rng = np.random.default_rng(seed)
    params = [[rng.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]
    if perm is not None:
        params[0][0] = params[0][0][perm]
    for _ in range(epochs):
        order = rng.permutation(len(X))
        for i in range(0, len(order), 64):
            idx = order[i : i + 64]
            x, (h1, h2, z) = X[idx], mlp_forward(params, X[idx])
            dz = softmax(z)
            dz[np.arange(len(idx)), y[idx]] -= 1
            dz /= len(idx)
            dh2 = (dz @ params[2][0].T) * (h2 > 0)
            dh1 = (dh2 @ params[1][0].T) * (h1 > 0)
            for layer, (gW, gb) in zip(
                params, [(x.T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]
            ):
                layer[0] -= 0.1 * gW
                layer[1] -= 0.1 * gb
    return params


def mlp_forward(params, X):
    """X: n × 784 (hoặc n × 28 × 28, sẽ được trải phẳng). Trả về h₁, h₂ và 10 logit."""
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, X.reshape(len(X), -1) @ W1 + b1)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


t0 = time.perf_counter()
MLP = train_mlp(Xtr.reshape(-1, 784), ytr)
acc_mlp = float((mlp_forward(MLP, Xte)[2].argmax(1) == yte).mean())
print(
    f"MLP: đúng {pct(acc_mlp)} trên tập kiểm tra ({vn(time.perf_counter() - t0, 1)} giây); tệp của demo ghi {pct(demo['grid']['mlp'][CENTER])}"
)
assert abs(acc_mlp - 0.9727) < 5e-4, "khác với demo: hãy kiểm tra lại dữ liệu"

Cùng công thức huấn luyện, cùng seed: đúng 97,27% như demo và như Bài 04.

## 9 · Vật dời thì bản đồ dời theo: tính đẳng biến

Hàm `shift` dưới dời nét chữ `dx` điểm ảnh sang phải và `dy` điểm ảnh xuống dưới (số âm là sang trái, lên trên),
lấp chỗ trống bằng 0, giống hệt `cnn.py` và demo.

In [ ]:
def shift(X, dx, dy):
    """Dời nét chữ dx điểm ảnh sang phải, dy điểm ảnh xuống (X: n × 28 × 28); điểm ảnh mới lộ ra bằng 0."""
    out = np.zeros_like(X)
    src = lambda d: slice(max(0, -d), 28 - max(0, d))
    dst = lambda d: slice(max(0, d), 28 - max(0, -d))
    out[:, dst(dy), dst(dx)] = X[:, src(dy), src(dx)]
    return out


def move_map(Z, dx, dy, fill):
    """Dời một bản đồ (n × 26 × 26 × F) đúng (dx, dy) ô; ô mới lộ ra nhận giá trị fill (một số cho mỗi bộ lọc)."""
    out = np.broadcast_to(fill, Z.shape).copy()
    H, W = Z.shape[1:3]
    out[:, max(0, dy) : H - max(0, -dy), max(0, dx) : W - max(0, -dx)] = Z[
        :, max(0, -dy) : H - max(0, dy), max(0, -dx) : W - max(0, dx)
    ]
    return out

### 🤔 Dự đoán trước
Dời chữ số 4 sang phải 3 điểm ảnh và xuống 2 điểm ảnh, rồi tính lại 8 bản đồ của CNN (trước ReLU). So với 8 bản đồ cũ
dời đúng (3, 2) ô, bao nhiêu trong 676 vị trí khớp hoàn toàn? Và CNN có còn đoán đúng số 4 không?

In [ ]:
dx, dy = 3, 2
x0, x1 = Xte[109:110], shift(Xte[109:110], dx, dy)
Z0, Z1 = conv2d(x0, CNN["Wc"], CNN["bc"]), conv2d(x1, CNN["Wc"], CNN["bc"])
want = move_map(Z0, dx, dy, CNN["bc"])  # ô mới lộ ra chỉ thấy nền 0, nên bằng đúng b
match = np.isclose(Z1, want, rtol=0, atol=1e-12).all(-1)[0]  # khớp ở cả 8 bộ lọc
exact = (Z1 == want).all(-1)[0]
print(f"{match.sum()}/676 vị trí khớp đến 10⁻¹² ở cả 8 bản đồ ({exact.sum()} khớp từng bit)")
p0, p1 = softmax(cnn_logits(CNN, x0))[0], softmax(cnn_logits(CNN, x1))[0]
print(
    f"CNN: ảnh gốc đoán {p0.argmax()} (p = {pct(p0.max())}); dời ({dx}, {dy}) đoán {p1.argmax()} (p = {pct(p1.max())})"
)
assert match.sum() == 676

Cả 676 vị trí đều khớp: bản đồ của ảnh dời **chính là** bản đồ cũ dời theo. Lý do nằm ngay trong công thức: mỗi ô
của bản đồ chỉ nhìn cửa sổ $3 \times 3$ bên dưới nó bằng cùng một bộ lọc, nên cửa sổ dời đi thì ô kết quả dời theo.
Tính chất này gọi là **đẳng biến** với phép tịnh tiến (translation equivariance): dời rồi tích chập bằng tích chập
rồi dời. Nó có sẵn trong cấu trúc, không cần học, và đúng chính xác với các điều kiện: dời nguyên số điểm ảnh, bước
trượt 1, và xa mép ảnh (nét bị đẩy ra ngoài thì mất; 10 ảnh mẫu đều có lề trống ≥ 4 điểm ảnh).

Nhưng CNN vẫn đổi câu trả lời: dời (3, 2), nó đoán số 2 thay vì 4. Đẳng biến của lớp tích chập
chưa phải là **bất biến** (invariance) của cả mạng: đầu ra không đổi khi vật dời. Ta kiểm tra trên cả 10 ảnh mẫu
và mọi phép dời trong $[-4, 4]^2$, rồi theo dõi tín hiệu qua từng bước sau tích chập.

In [ ]:
moves = [(a, b) for b in range(-4, 5) for a in range(-4, 5) if (a, b) != (0, 0)]
Xs = Xte[SAMPLES]
Zs = conv2d(Xs, CNN["Wc"], CNN["bc"])
n_ok = 0
for a, b in moves:
    ok = np.isclose(conv2d(shift(Xs, a, b), CNN["Wc"], CNN["bc"]), move_map(Zs, a, b, CNN["bc"]), rtol=0, atol=1e-12)
    n_ok += int(ok.all(axis=(1, 2, 3)).sum())  # ảnh nào khớp ở cả 676 vị trí × 8 bộ lọc
print(f"tích chập: đẳng biến ở {n_ok}/{len(moves) * len(SAMPLES)} cặp (ảnh mẫu, phép dời)")
assert n_ok == len(moves) * len(SAMPLES)

H0 = maxpool2(np.maximum(Zs, 0))  # 10 × 13 × 13 × 8 sau ReLU và pooling
for k in range(1, 5):
    Hk = maxpool2(np.maximum(conv2d(shift(Xs, k, 0), CNN["Wc"], CNN["bc"]), 0))
    if k % 2 == 0:  # dời chẵn: bản đồ sau pooling có dời đúng k/2 ô không?
        same = np.array_equal(Hk[:, :, k // 2 :], H0[:, :, : 13 - k // 2])
        print(f"dời {k} điểm ảnh sang phải: bản đồ sau pooling dời đúng {k // 2} ô? {same}")
    else:
        best = min(
            np.linalg.norm(Hk[:, :, c:] - H0[:, :, : 13 - c]) / np.linalg.norm(H0[:, :, : 13 - c]) for c in (0, 1)
        )
        print(f"dời {k} điểm ảnh sang phải: bản đồ sau pooling khác mọi bản dời nguyên ô của bản cũ ≥ {pct(best, 0)}")

Ba bước sau tích chập phản ứng khác nhau với phép dời:
- Tích chập: đẳng biến ở cả 800 cặp (10 ảnh mẫu × 80 phép dời khác 0).
- Max-pooling 2×2: dời **chẵn** điểm ảnh thì bản đồ nhỏ dời đúng nửa số ô, vẫn đẳng biến. Dời **lẻ** thì các khối
  $2 \times 2$ bị cắt ở chỗ khác, và bản đồ sau pooling khác mọi bản dời nguyên ô của bản cũ ít nhất 53%. Pooling chỉ
  bỏ qua được những dịch chuyển nhỏ *bên trong* một khối, và không phải lúc nào cũng vậy.
- Lớp dense: gắn một trọng số riêng với từng ô trong 13 × 13 × 8 ô. Bản đồ dời một ô là đưa tín hiệu vào những
  trọng số khác hẳn, nên lớp này không hề đẳng biến.

Hình dưới cho thấy bước đầu tiên. Hàng trên là chữ số 4 gốc, hàng dưới là nó dời (3, 2). Cột giữa là bản đồ của bộ lọc
#4 (giống Sobel dọc), với vòng tròn xanh lá ở ô lớn nhất; ở hàng dưới, ô vuông nét đứt là vị trí cũ của ô đó.
Cột phải: trên là bản đồ cũ dời (3, 2) ô, dưới là hiệu giữa nó và bản đồ mới, xám nhạt đều (bằng 0) ở mọi ô.
Nền của các bản đồ có màu cam nhạt vì cửa sổ toàn nền 0 cho đúng $z = b$, mà $b$ của bộ lọc #4 âm.

In [ ]:
f4 = 3  # bộ lọc #4
fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.2))
peak0 = np.unravel_index(Z0[0, ..., f4].argmax(), (M, M))
for row, (x, Z, title) in enumerate([(x0, Z0, "ảnh gốc"), (x1, Z1, f"dời ({dx}, {dy})")]):
    axes[row, 0].imshow(x[0], cmap="gray_r", vmin=0, vmax=1)
    axes[row, 0].set_title(title, fontsize=10)
    axes[row, 0].axis("off")
    peak = np.unravel_index(Z[0, ..., f4].argmax(), (M, M))
    show_map(axes[row, 1], Z[0, ..., f4], f"bộ lọc #4: lớn nhất {vn(Z[0, ..., f4].max(), 2)} ở {tuple(map(int, peak))}")
    axes[row, 1].plot(peak[1], peak[0], "o", ms=14, mfc="none", mec=GREEN, mew=2.5)
    if row == 1:
        axes[row, 1].add_patch(Rectangle((peak0[1] - 0.5, peak0[0] - 0.5), 1, 1, fill=False, ec=GREEN, lw=1.5, ls="--"))
diff = Z1[0, ..., f4] - want[0, ..., f4]
show_map(axes[0, 2], move_map(Z0, dx, dy, CNN["bc"])[0, ..., f4], f"bản đồ cũ dời ({dx}, {dy}) ô")
show_map(axes[1, 2], diff, f"hiệu: bằng 0 ở {(diff == 0).sum()}/676 ô", vmax=np.abs(Z0[0, ..., f4]).max())
plt.tight_layout()
plt.show()

## 10 · Dời cả 10.000 ảnh kiểm tra

Một ảnh chưa nói được nhiều. Ta dời **mọi** ảnh kiểm tra $k$ điểm ảnh, $k = 0 \dots 4$, theo 4 hướng (trái, phải, lên,
xuống), rồi lấy trung bình độ chính xác của 4 hướng, như biểu đồ của demo.

Tính đẳng biến cho ta một lối tắt: thay vì dời ảnh rồi tích chập 17 lần, ta đệm mỗi ảnh thêm 4 điểm ảnh 0 mỗi bên
(36 × 36), tích chập **một lần** (bản đồ 34 × 34), rồi cắt ra cửa sổ 26 × 26 ứng với từng phép dời. Ô dưới kiểm tra
lối tắt này cho ra đúng từng bit như cách làm thẳng, rồi chạy CNN của demo và MLP vừa huấn luyện trên 17 phép dời.

In [ ]:
AXIS = [(0, 0)] + [d for k in range(1, 5) for d in [(k, 0), (-k, 0), (0, k), (0, -k)]]  # 17 phép dời trên hai trục


def dense_head(P, A):
    return maxpool2(np.maximum(A, 0)).reshape(len(A), -1) @ P["Wd"] + P["bd"]


Xpad = np.pad(Xte[:50], ((0, 0), (4, 4), (4, 4)))
Apad = conv2d(Xpad, CNN["Wc"], CNN["bc"])  # 50 × 34 × 34 × 8
assert all(
    np.array_equal(Apad[:, 4 - b : 30 - b, 4 - a : 30 - a], conv2d(shift(Xte[:50], a, b), CNN["Wc"], CNN["bc"]))
    for a, b in AXIS
)

t0 = time.perf_counter()
right = {"cnn": {}, "mlp": {}}  # đúng/sai của từng ảnh kiểm tra, cho mỗi phép dời
for i in range(0, len(Xte), 200):
    Apad = conv2d(np.pad(Xte[i : i + 200], ((0, 0), (4, 4), (4, 4))), CNN["Wc"], CNN["bc"])
    for a, b in AXIS:
        ok = dense_head(CNN, Apad[:, 4 - b : 30 - b, 4 - a : 30 - a]).argmax(1) == yte[i : i + 200]
        right["cnn"].setdefault((a, b), []).append(ok)
for a, b in AXIS:
    right["cnn"][(a, b)] = np.concatenate(right["cnn"][(a, b)])
    right["mlp"][(a, b)] = mlp_forward(MLP, shift(Xte, a, b))[2].argmax(1) == yte
axis_acc = {m: [np.mean([right[m][d].mean() for d in AXIS if max(map(abs, d)) == k]) for k in range(5)] for m in right}
print(f"17 phép dời × 10.000 ảnh × 2 mạng ({vn(time.perf_counter() - t0, 1)} giây)")
for m, name in [("mlp", "MLP"), ("cnn", "CNN seed 0")]:
    print(f"{name:>10}: " + " · ".join(f"k = {k}: {pct(v)}" for k, v in enumerate(axis_acc[m])))
assert np.allclose(axis_acc["cnn"], demo["axis"]["cnn"][0], atol=5e-4), "khác với demo: hãy kiểm tra lại dữ liệu"
assert np.allclose(axis_acc["mlp"], demo["axis"]["mlp"], atol=5e-4), "khác với demo: hãy kiểm tra lại dữ liệu"

Lối tắt cho ra đúng từng bit, và độ chính xác của cả hai mạng trùng với tệp của demo ở mọi mức dời. Hình dưới vẽ
chúng cùng hai kết quả chỉ có trong tệp (cần huấn luyện lâu): CNN seed 1 và 2 (vạch dọc xanh dương nối thấp nhất và cao
nhất của 3 seed), và một MLP được huấn luyện với mỗi ảnh dời ngẫu nhiên tới ±2 điểm ảnh (tăng cường dữ liệu, Bài 08).
MLP: vòng rỗng xám, nét chấm; CNN: chấm tròn xanh dương, nét liền; MLP + dời: ô vuông cam, nét gạch.

In [ ]:
ks = np.arange(5)
cnn_seeds = np.array(demo["axis"]["cnn"])  # 3 seed × 5 mức dời (seed 0 vừa được tính lại ở trên)
aug = np.array(demo["axis"]["mlpAug"])
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(ks, 100 * np.array(axis_acc["mlp"]), color=GREY, ls=":", lw=2, marker="o", mfc="white", ms=8, label="MLP")
ax.vlines(ks, 100 * cnn_seeds.min(0), 100 * cnn_seeds.max(0), color=BLUE, lw=2)
ax.plot(ks, 100 * np.median(cnn_seeds, 0), color=BLUE, lw=2, marker="o", ms=7, label="CNN (trung vị 3 seed)")
ax.plot(ks, 100 * aug, color=ORANGE, ls="--", lw=2, marker="s", ms=7, label="MLP học với ảnh dời ±2")
ax.axhline(10, color="#555", lw=1, ls="--")
ax.text(4, 12, "đoán mò 10%", ha="right", fontsize=9, color="#555")
ax.set(
    xlabel="số điểm ảnh dời (trung bình 4 hướng)", ylabel="đúng trên 10.000 ảnh kiểm tra (%)", ylim=(0, 100), xticks=ks
)
ax.set_title("Dời ảnh kiểm tra: CNN bền hơn MLP nhưng không bất biến")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="y", alpha=0.3)
vn_axes(fig)
plt.show()
for k in ks:
    print(
        f"dời {k}: MLP {pct(axis_acc['mlp'][k])} · CNN {pct(cnn_seeds[:, k].min())}–{pct(cnn_seeds[:, k].max())} (3 seed)"
        f" · MLP + dời ±2 {pct(aug[k])}"
    )

Dời 1 điểm ảnh, cả hai mạng chỉ mất vài điểm (CNN 95,40–96,27% qua 3 seed, MLP 94,39%). Dời 4 điểm ảnh, CNN chỉ còn
36,78–42,46%: bền hơn MLP (29,49%) nhưng còn xa mới bất biến. Còn MLP được huấn luyện với ảnh dời ngẫu nhiên giữ
75,33% ở 4 điểm ảnh, hơn mọi seed của CNN từ mức dời 1. Sự bền vững với phép dời có thể **học từ dữ liệu**; CNN
có sẵn một phần trong cấu trúc, với ít tham số hơn 8 lần. Ở mức 0, MLP + dời (96,85%) thấp hơn cả 3 seed CNN
(97,40–97,98%), nên không mạng nào thắng ở mọi mức. (Mỗi đường MLP chỉ có một lần chạy, seed 0.)

### 🤔 Dự đoán trước
Dời 4 điểm ảnh, CNN seed 0 chỉ còn 41,09% số ảnh đúng. Có phải chủ yếu vì một phần nét bị đẩy ra khỏi khung 28×28?
Nếu chỉ giữ những ảnh **không mất chút mực nào** khi dời (4 hàng hoặc 4 cột ở phía bị đẩy ra đều trống), độ chính xác
có trở lại gần 98% không?

In [ ]:
ink = test_img > 0
edge = {(4, 0): ink[:, :, 24:], (-4, 0): ink[:, :, :4], (0, 4): ink[:, 24:, :], (0, -4): ink[:, :4, :]}
names = {(4, 0): "phải", (-4, 0): "trái", (0, 4): "xuống", (0, -4): "lên"}
intact_acc = {}
for d, strip in edge.items():
    lost = strip.any(axis=(1, 2))  # ảnh có mực ở dải sẽ bị đẩy ra ngoài
    ok = right["cnn"][d]
    intact_acc[d] = ok[~lost].mean()
    print(
        f"dời 4 sang {names[d]:>5}: {pct(lost.mean()):>6} ảnh mất mực · CNN đúng {pct(ok.mean())} trên mọi ảnh, "
        f"{pct(intact_acc[d])} trên {vn((~lost).sum(), 0)} ảnh còn nguyên mực"
    )
assert max(intact_acc.values()) < 0.6

Không. Kể cả khi không mất chút mực nào, CNN chỉ đúng 30,95% đến 51,01% tùy hướng. Mất mực có làm hại khi dời lên
hay xuống (độ chính xác trên ảnh còn nguyên mực cao hơn trên mọi ảnh), nhưng khi dời sang trái hay phải thì ảnh mất
mực lại được đoán đúng nhiều hơn. Lý do chính là **vị trí**: mỗi chữ số MNIST được dời sao cho trọng tâm của nét nằm
đúng giữa khung 28×28 ([trang gốc của MNIST](https://web.archive.org/web/20231215034634/http://yann.lecun.com/exdb/mnist/)),
nên lớp dense chỉ từng thấy nét chữ quanh vị trí đó. Dời đi 4 điểm ảnh, bản đồ đặc
trưng vẫn dời theo đúng như trước, nhưng tín hiệu rơi vào những trọng số của lớp dense hầu như chưa từng được dùng.

Demo cho bạn dời từng ảnh mẫu và xem cả hai mạng trả lời gì. Ô dưới tính lại những câu trả lời đó cho cả 81 phép dời
trong $[-4, 4]^2$: dự đoán của MLP, xác suất của nó, và số nơ-ron trong 128 nơ-ron lớp đầu đổi trạng thái bật/tắt
so với ảnh gốc (tệp của demo lưu sẵn ba số này), rồi in kết quả ở phép dời (+4, 0).

In [ ]:
mism, wrong4, flips4 = 0, {"MLP": 0, "CNN": 0}, []
for s, i in zip(demo["samples"], SAMPLES):
    moved = np.stack([shift(Xte[i : i + 1], a, b)[0] for a, b in SHIFTS])  # 81 ảnh
    h1, _, z = mlp_forward(MLP, moved)
    p_mlp, p_cnn = softmax(z), softmax(cnn_logits(CNN, moved))
    flips = ((h1 > 0) != (h1[CENTER] > 0)).sum(1)
    for k, (pred, p, fl) in enumerate(s["mlp"]):
        mism += pred != p_mlp[k].argmax() or abs(p - p_mlp[k].max()) > 2e-3 or fl != flips[k]
    k = SHIFTS.index((4, 0))
    wrong4["MLP"] += p_mlp[k].argmax() != yte[i]
    wrong4["CNN"] += p_cnn[k].argmax() != yte[i]
    flips4.append(flips[k])
    print(
        f"#{i:>4} (số {yte[i]}) dời (+4, 0): MLP đoán {p_mlp[k].argmax()} ({pct(p_mlp[k].max(), 1)}, {flips[k]}/128 nơ-ron đổi)"
        f" · CNN đoán {p_cnn[k].argmax()} ({pct(p_cnn[k].max(), 1)})"
    )
print(
    f"dời (+4, 0): MLP sai {wrong4['MLP']}/10, CNN sai {wrong4['CNN']}/10; MLP đổi {min(flips4)}–{max(flips4)}/128 nơ-ron"
)
print(f"khác với tệp của demo: {mism} trên {len(SAMPLES) * len(SHIFTS)} cặp (ảnh, phép dời)")
assert mism == 0, "khác với demo: hãy kiểm tra lại dữ liệu"

Ở phép dời (+4, 0), MLP sai 9 trên 10 ảnh mẫu và 29 đến 74 trong 128 nơ-ron lớp đầu của nó đổi trạng thái; CNN cũng
sai 7 ảnh. Nhiều câu trả lời sai vẫn đi kèm xác suất cao (CNN đoán số 7 là 5 với 96,2%): mạng không biết mình đang
nhìn một ảnh lạ. Đây là các ảnh có lề rộng, nên không ảnh nào mất mực khi dời.

## 11 · Xáo trộn vị trí điểm ảnh

CNN tốt hơn nhờ đâu? Một phép thử gọn: chọn **một** hoán vị cố định của 784 vị trí (seed 123, như `cnn.py`), xáo
trộn mọi ảnh huấn luyện và kiểm tra theo đúng hoán vị đó, rồi huấn luyện lại. Ảnh sau khi xáo trộn trông như nhiễu
(hình dưới), nhưng không mất thông tin nào: hoán vị ngược lại là trả về ảnh gốc.

In [ ]:
PERM = np.random.default_rng(123).permutation(784)
Xtr_perm = Xtr.reshape(-1, 784)[:, PERM].reshape(-1, 28, 28)
Xte_perm = Xte.reshape(-1, 784)[:, PERM].reshape(-1, 28, 28)
fig, axes = plt.subplots(2, 5, figsize=(8, 3.4))
for col, i in enumerate(SAMPLES[:5]):
    for row, (X, name) in enumerate([(Xte, "gốc"), (Xte_perm, "xáo trộn")]):
        axes[row, col].imshow(X[i], cmap="gray_r", vmin=0, vmax=1)
        axes[row, col].set_title(f"số {yte[i]}, {name}", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()
assert np.array_equal(Xte_perm.reshape(-1, 784)[:, np.argsort(PERM)], Xte.reshape(-1, 784))

### 🤔 Dự đoán trước
Huấn luyện lại MLP và CNN trên ảnh đã xáo trộn, cùng công thức. Mạng nào mất nhiều độ chính xác hơn? Có mạng nào
không mất gì không?

Với MLP, ta trả lời được chính xác. MLP tính $xW_1$: xáo trộn các phần tử của $x$ chỉ tương đương **đổi thứ tự các
hàng** của $W_1$. Nếu khởi tạo $W_1$ cũng xáo trộn hàng theo cùng hoán vị, hai mạng phải giống hệt nhau ở mọi bước.
Ô dưới kiểm tra điều đó trên một lượt 10.000 ảnh, rồi huấn luyện lại CNN trên ảnh xáo trộn (313 bước như mục 7).

In [ ]:
Xf, Xf_perm = Xtr.reshape(-1, 784)[:10_000], Xtr_perm.reshape(-1, 784)[:10_000]
mlp_a = train_mlp(Xf, ytr[:10_000], epochs=1)
mlp_b = train_mlp(Xf_perm, ytr[:10_000], epochs=1, perm=PERM)
za, zb = mlp_forward(mlp_a, Xte)[2], mlp_forward(mlp_b, Xte_perm)[2]
print(
    f"MLP 1 lượt, 10.000 ảnh: gốc {pct((za.argmax(1) == yte).mean())}, xáo trộn {pct((zb.argmax(1) == yte).mean())};"
    f" logit lệch nhau tối đa {sci(np.abs(za - zb).max())}"
)
assert np.array_equal(za.argmax(1), zb.argmax(1)) and np.allclose(za, zb, atol=1e-10)

P_perm, _ = train_cnn(seed=0, X=Xtr_perm)
acc_perm = accuracy(P_perm, Xte_perm)
print(f"CNN 313 bước, seed 0: gốc {pct(acc_live)}, xáo trộn {pct(acc_perm)}")
print(
    f"công thức đủ (tệp của demo, seed 0): MLP {pct(demo['grid']['mlp'][CENTER])} → {pct(demo['perm']['mlp'])}, "
    f"CNN {pct(demo['grid']['cnn'][CENTER])} → {pct(demo['perm']['cnn'])}"
)
print(f"CNN trên ảnh gốc qua 3 seed: {pct(cnn_seeds[:, 0].min())}–{pct(cnn_seeds[:, 0].max())}")

Với MLP, xáo trộn điểm ảnh **không thay đổi gì**: cùng 80,90%, logit trùng nhau đến sai số làm tròn. MLP không hề biết
điểm ảnh nào nằm cạnh điểm ảnh nào, nên nó cũng không mất gì khi thứ tự bị phá. Trong `cnn.py`, MLP trên ảnh xáo trộn
được khởi tạo bằng $W_1$ *không* xáo trộn hàng, tức là một khởi tạo ngẫu nhiên khác, nên nó ra 97,50% thay vì 97,27%:
chênh lệch đó chỉ do khởi tạo.

CNN thì mất: 92,66% → 90,33% sau 313 bước, và 97,98% → 95,09% với công thức đủ. Mức 95,09% nằm xa dưới khoảng
97,40–97,98% của 3 seed, nên đây không phải dao động giữa các seed. Trên ảnh xáo trộn, một cửa sổ $3 \times 3$ gom 9
điểm ảnh nằm rải rác khắp chữ số, nên “mẫu cục bộ” không còn ý nghĩa, và CNN thua cả MLP. Lợi thế của CNN đến từ
một **giả định** về dữ liệu (thiên kiến quy nạp, *inductive bias*): điểm ảnh nằm gần nhau thì liên quan với nhau,
và một mẫu có ích ở chỗ này thì cũng có ích ở chỗ khác. Ảnh thật thỏa giả định đó; ảnh xáo trộn thì không.

## 12 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Bộ lọc có cần học không?
Mục 5 cho thấy 8 bộ lọc học được vẫn gần với chính chúng lúc khởi tạo. Vậy nếu **giữ nguyên** bộ lọc ngẫu nhiên và chỉ
học lớp dense thì sao? `learn_filters=False` làm đúng việc đó, với cùng seed (nên cùng bộ lọc ban đầu) và cùng 313 bước.

In [ ]:
SEEDS = (0, 1, 2)
res = {"học": [], "cố định": []}
for seed in SEEDS:
    P_learn = P_live if seed == 0 else train_cnn(seed=seed)[0]  # seed 0 đã huấn luyện ở mục 7
    res["học"].append(accuracy(P_learn))
    res["cố định"].append(accuracy(train_cnn(seed=seed, learn_filters=False)[0]))
    print(f"seed {seed}: bộ lọc học được {pct(res['học'][-1])} · bộ lọc ngẫu nhiên cố định {pct(res['cố định'][-1])}")
for name, v in res.items():
    print(f"{name:>8}: {pct(min(v))}–{pct(max(v))}")
gap = 100 * (np.array(res["học"]) - np.array(res["cố định"]))
print("học bộ lọc hơn giữ nguyên (điểm %, theo seed):", " · ".join(vn(g, 2) for g in gap))

Ở cả 3 seed, học bộ lọc tốt hơn giữ nguyên, nhưng chỉ hơn 0,56 đến 0,99 điểm. Chênh lệch đó nhỏ hơn dao động giữa
các seed (90,45–92,66% với bộ lọc học được) và hai khoảng 3 seed chồng lên nhau; ta chỉ thấy nó vì so từng cặp cùng
seed (cùng bộ lọc ban đầu, cùng thứ tự ảnh). Phần lớn lợi ích đến từ **kiến trúc**: cửa sổ cục bộ, trọng số dùng
chung, pooling. 8 bộ lọc ngẫu nhiên đã nhìn theo 8 hướng khác nhau trong không gian 9 chiều của các mẫu $3 \times 3$,
và lớp dense học cách kết hợp chúng. Saxe và cộng sự (2011) phân tích vì sao các kiến trúc tích chập + pooling với
trọng số ngẫu nhiên vẫn cho đặc trưng nhận dạng tốt đến bất ngờ. Kết luận này chỉ đúng cho mạng một lớp nhỏ này:
bộ lọc ở các lớp sâu của CNN lớn ghép nhiều đặc trưng lại với nhau, và chúng phải được học.

### Bộ lọc của riêng bạn, và điều kiện của tính đẳng biến
Sửa `K_MINE` thành bất kỳ bộ lọc $3 \times 3$ nào. Ô dưới vẽ bản đồ của nó trên chữ số 4, rồi kiểm tra tính đẳng
biến với 1.000 bộ lọc ngẫu nhiên và hai cách trượt: bước 1 (như trên) và **bước 2** (chỉ giữ mỗi ô thứ hai theo mỗi
chiều, như một số mạng làm để thu nhỏ bản đồ).

In [ ]:
K_MINE = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], float)  # thử sửa: đây là cạnh ngang “hàng trên trừ hàng dưới”
fig, (a, b) = plt.subplots(1, 2, figsize=(6.4, 3.2))
a.imshow(I, cmap="gray_r", vmin=0, vmax=1)
a.set_title("chữ số 4 (#109)", fontsize=10)
a.axis("off")
im = show_map(b, conv2d(I, K_MINE.reshape(9, 1))[..., 0], "bản đồ của K_MINE")
fig.colorbar(im, ax=b, fraction=0.046)
vn_axes(fig)
plt.show()

W_rand = np.random.default_rng(5).normal(size=(9, 1000))  # 1.000 bộ lọc ngẫu nhiên
x0, x1, x2 = Xte[109:110], shift(Xte[109:110], 1, 0), shift(Xte[109:110], 2, 0)
Z0, Z1, Z2 = (conv2d(x, W_rand) for x in (x0, x1, x2))
print("bước 1, dời 1 điểm ảnh: đẳng biến?", np.allclose(Z1, move_map(Z0, 1, 0, 0.0), atol=1e-12))
S0, S1, S2 = Z0[:, ::2, ::2], Z1[:, ::2, ::2], Z2[:, ::2, ::2]  # bước trượt 2: bản đồ 13 × 13
print("bước 2, dời 2 điểm ảnh: bản đồ dời đúng 1 ô?", np.allclose(S2[:, :, 1:], S0[:, :, :-1], atol=1e-12))
rel = np.linalg.norm(S1[:, :, 1:] - S0[:, :, :-1]) / np.linalg.norm(S0[:, :, :-1])
print(
    f"bước 2, dời 1 điểm ảnh: khác bản dời 1 ô {pct(rel, 0)}, khác bản không dời "
    f"{pct(np.linalg.norm(S1 - S0) / np.linalg.norm(S0), 0)}"
)

Với bước trượt 1, mọi bộ lọc đều đẳng biến: tính chất này đến từ cấu trúc của phép tính, không từ trọng số. Với bước
trượt 2, chỉ phép dời **chẵn** còn được bảo toàn; dời 1 điểm ảnh cho một bản đồ không khớp với bản cũ, dù dời 0 hay
1 ô, giống max-pooling ở mục 9. Đây là lý do Azulay & Weiss (2018) và Zhang (2019) thấy các CNN hiện đại có thể đổi
câu trả lời khi ảnh dời chỉ 1 điểm ảnh: mọi bước thu nhỏ (pooling, bước trượt > 1) đều phá tính đẳng biến với những
phép dời không chia hết cho bước.

## 13 · Giới hạn và cầu nối: vùng tiếp nhận

Mỗi ô của bản đồ chỉ nhìn một cửa sổ $3 \times 3$: đó là **vùng tiếp nhận** (receptive field) của nó, phần ảnh có
thể làm giá trị của ô thay đổi. Sau max-pooling 2×2, một ô nhìn $4 \times 4$ điểm ảnh. Trong mạng của demo, chỉ lớp
dense nhìn thấy cả ảnh, và nó gắn chặt với vị trí. Muốn có đặc trưng nhìn được vùng lớn mà vẫn dùng chung trọng số,
ta **xếp chồng** nhiều lớp tích chập và pooling: mỗi lớp nhìn bản đồ của lớp trước, nên vùng tiếp nhận lớn dần.

Ô dưới đo vùng tiếp nhận bằng thực nghiệm. Cho 784 ảnh, mỗi ảnh chỉ có **một** điểm ảnh bằng 1, qua một chồng lớp
(bộ lọc toàn số 1, nên không có số nào triệt tiêu nhau), rồi đếm những điểm ảnh làm ô ở giữa bản đồ cuối khác 0.

In [ ]:
def receptive_field(layers):
    """Cạnh vùng tiếp nhận của ô giữa bản đồ cuối: mỗi lớp là 'conv' (3×3, bước 1) hoặc 'pool' (2×2, bước 2)."""
    A = np.eye(784).reshape(784, 28, 28, 1)  # 784 ảnh, mỗi ảnh một điểm ảnh sáng
    for layer in layers:
        if layer == "conv":
            A = conv2d(A[..., 0], np.ones((9, 1)))
        else:  # cạnh lẻ thì bỏ hàng/cột cuối, như các thư viện làm
            A = maxpool2(A[:, : A.shape[1] // 2 * 2, : A.shape[2] // 2 * 2])
    seen = A[:, A.shape[1] // 2, A.shape[2] // 2, 0].reshape(28, 28) > 0
    rows, cols = np.nonzero(seen)
    side = int(np.ptp(rows)) + 1
    assert seen.sum() == side * (int(np.ptp(cols)) + 1)  # vùng nhìn thấy là một hình chữ nhật liền
    return side


stacks = {
    "chỉ tích chập": ["conv"] * 5,
    "tích chập + pooling": ["conv", "pool", "conv", "pool", "conv"],
}
for name, layers in stacks.items():
    sizes = [receptive_field(layers[: n + 1]) for n in range(len(layers))]
    print(f"{name:>20}: " + " → ".join(f"{L} {s}×{s}" for L, s in zip(layers, sizes)))
assert receptive_field(["conv", "pool"]) == 4 and receptive_field(stacks["tích chập + pooling"]) == 18

Mỗi lớp tích chập 3×3 nới vùng tiếp nhận thêm 2 điểm ảnh, còn mỗi lớp pooling nhân đôi bước nhảy của các lớp sau.
Năm lớp tích chập liền nhau mới nhìn được 11×11; xen pooling vào thì cùng 5 lớp đã nhìn 18×18, gần cả chữ số.
Ở những CNN lớn huấn luyện trên ImageNet, Zeiler & Fergus (2013) quan sát thấy lớp đầu phản ứng với cạnh và màu,
lớp 2 với góc và tổ hợp cạnh–màu, lớp 3 với hoa văn, các lớp sau với bộ phận rồi cả đối tượng. Vùng tiếp nhận
lớn dần là điều kiện để có thứ bậc đó; mạng một lớp của bài này chưa có.

Mạng của bài này còn hai giới hạn đã đo được:
- **Chưa bất biến:** dời 4 điểm ảnh, CNN chỉ còn 36,78–42,46% (3 seed), vì lớp dense phía sau gắn với vị trí.
- **Pooling không miễn phí:** max-pooling đẳng biến với phép dời chẵn nhưng không với phép dời lẻ.

[Bài 12 · Pooling & kiến trúc CNN kinh điển](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/kien-truc-cnn/) xếp chồng
tích chập và pooling thành LeNet, so các kiểu pooling, và đo lại đúng phép thử dời ảnh này.

## Tóm tắt
- Tích chập trượt một bộ lọc $k \times k$ trên ảnh: mỗi ô là tổng $k^2$ tích cộng $b$. Ảnh 28×28, bộ lọc 3×3 cho bản
  đồ 26×26 (6.084 MAC). Viết lại bằng im2col, nó là một phép nhân ma trận $Z = PW + b$.
- Thư viện deep learning tính tương quan chéo (không lật bộ lọc) và gọi là tích chập; với bộ lọc học được, hai cách
  chỉ khác nhau một phép lật.
- Sobel dọc/ngang sáng ở mép nét. CNN của demo tự học 8 bộ lọc; 4 cái giống bộ dò cạnh (|cos| ≥ 0,75), dù cái nào
  cũng vẫn cùng hướng với chính nó lúc khởi tạo.
- CNN: 13.610 tham số, đúng 97,98% (MLP: 109.386 tham số, 97,27%). 80 tham số tích chập tốn 48.672 MAC, gấp 3,6 lần
  lớp dense, vì được dùng lại ở 676 vị trí.
- Tích chập đẳng biến chính xác với phép dời nguyên điểm ảnh (676/676 ô; 800/800 cặp ảnh–phép dời). Max-pooling và
  bước trượt 2 chỉ giữ được điều đó với phép dời chẵn; lớp dense thì không.
- Dời 4 điểm ảnh: MLP 29,49%, CNN 36,78–42,46%, MLP học với ảnh dời ±2 được 75,33%. Đẳng biến chưa phải bất biến.
- Xáo trộn điểm ảnh: MLP không đổi (chỉ là hoán vị hàng của $W_1$), CNN tụt từ 97,98% xuống 95,09%. Lợi thế của CNN là
  giả định rằng điểm ảnh gần nhau thì liên quan với nhau.

## Câu hỏi ôn tập
1. Ảnh 28×28, bộ lọc 5×5, bước trượt 1, không đệm: bản đồ đặc trưng có cỡ bao nhiêu, và tốn bao nhiêu MAC mỗi bộ lọc?
2. Một bộ lọc 3×3 có toàn bộ trọng số bằng nhau (1/9). Nó làm gì với ảnh, và bản đồ của nó ở mép một nét dọc ra sao?
3. Vì sao lớp tích chập có ít tham số hơn lớp dense hơn 100 lần mà lại tốn nhiều phép tính hơn?
4. Dời ảnh 2 điểm ảnh sang phải. Bản đồ sau tích chập thay đổi thế nào? Sau max-pooling 2×2? Còn dời 1 điểm ảnh?
5. Vì sao xáo trộn vị trí điểm ảnh (cùng một cách cho mọi ảnh) không ảnh hưởng gì tới MLP?
6. Một mạng chỉ gồm các lớp tích chập 3×3 (bước 1, không pooling, đệm 0 ở mép để bản đồ giữ cỡ 28×28) cần bao nhiêu
   lớp để ô ở giữa có vùng tiếp nhận rộng ít nhất 28×28?

<details><summary>Gợi ý đáp án</summary>

1. $(28 - 5 + 1)^2 = 24 \times 24 = 576$ ô, mỗi ô 25 MAC: $576 \cdot 25 = 14.400$ MAC.
2. Nó lấy trung bình 9 điểm ảnh, tức làm mờ ảnh. Ở mép một nét, giá trị chuyển dần từ 0 sang giá trị của nét thay vì
   nhảy bậc; bản đồ không đổi dấu như bộ lọc Sobel, vì mọi trọng số đều dương.
3. Mỗi trọng số tích chập được dùng lại ở 676 vị trí (mục 6): số tham số đếm trọng số khác nhau, số MAC đếm mỗi lần
   dùng. Lớp dense dùng mỗi trọng số đúng một lần cho mỗi ảnh.
4. Sau tích chập: bản đồ dời đúng 2 ô (đẳng biến, nếu nét không bị đẩy ra khỏi khung). Sau pooling 2×2: dời đúng 1 ô.
   Dời 1 điểm ảnh: bản đồ tích chập dời 1 ô, nhưng sau pooling các khối 2×2 bị cắt khác đi, nên bản đồ nhỏ không bằng
   bản dời nào của bản cũ.
5. $x$ bị xáo trộn thì $xW_1$ chỉ tương đương $W_1$ bị đổi thứ tự hàng; MLP học được $W_1$ đó như bất kỳ $W_1$ nào khác
   (mục 11 kiểm tra: khởi tạo đổi hàng tương ứng thì cho ra đúng cùng một mạng).
6. Mỗi lớp nới thêm 2 điểm ảnh: $1 + 2L \ge 28$ nên cần $L = 14$ lớp (vùng 29×29). Không đệm thì bản đồ đã co về
   2×2 sau 13 lớp. Xen pooling vào thì cần ít lớp hơn nhiều (mục 13).
</details>

## Tìm hiểu thêm
- Dive into Deep Learning, [*Convolutions for Images*](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html):
  hàm `corr2d` bằng vòng lặp, bộ lọc cạnh, học bộ lọc từ dữ liệu, tương quan chéo và vùng tiếp nhận.
- CS231n, [*Convolutional Neural Networks*](https://cs231n.github.io/convolutional-networks/): chia sẻ trọng số
  (ví dụ AlexNet), im2col, công thức $(W - F + 2P)/S + 1$.
- Goodfellow, Bengio & Courville, [*Deep Learning*, Chương 9](https://www.deeplearningbook.org/contents/convnets.html):
  tương tác thưa, chia sẻ tham số, đẳng biến, pooling.
- V. Dumoulin & F. Visin (2016), [A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285).
- K. Fukushima (1980), [Neocognitron](https://doi.org/10.1007/BF00344251); Y. LeCun và cộng sự (1989),
  [Backpropagation applied to handwritten zip code recognition](https://doi.org/10.1162/neco.1989.1.4.541) và (1998),
  [Gradient-based learning applied to document recognition](https://doi.org/10.1109/5.726791).
- M. Zeiler & R. Fergus (2013), [Visualizing and Understanding Convolutional Networks](https://arxiv.org/abs/1311.2901);
  C. Olah và cộng sự (2017), [Feature Visualization](https://distill.pub/2017/feature-visualization/).
- A. Saxe và cộng sự (2011), [On random weights and unsupervised feature learning](http://www.icml-2011.org/papers/551_icmlpaper.pdf):
  vì sao kiến trúc tích chập + pooling với trọng số ngẫu nhiên vẫn cho đặc trưng tốt.
- A. Azulay & Y. Weiss (2018), [Why do deep convolutional networks generalize so poorly to small image transformations?](https://arxiv.org/abs/1805.12177);
  R. Zhang (2019), [Making Convolutional Networks Shift-Invariant Again](https://arxiv.org/abs/1904.11486);
  O. Kayhan & J. van Gemert (2020), [On Translation Invariance in CNNs](https://arxiv.org/abs/2003.07064).
- Công cụ tương tác: [CNN Explainer](https://poloclub.github.io/cnn-explainer/), [Image Kernels](https://setosa.io/ev/image-kernels/).

**Bài tiếp theo:** [Bài 12 · Pooling & kiến trúc CNN kinh điển](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/kien-truc-cnn/):
xếp chồng tích chập và pooling thành LeNet, và đo xem mỗi kiểu thu nhỏ bản đồ giữ được tính đẳng biến đến đâu.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")